# Цепочки задержек рейсов

**Проект на pandas и PostgreSQL**

Как часто самолёт задерживается на следующем вылете, если предыдущий рейс
прилетел поздно? Проверим, как это связано с плановым временем между рейсами.

- **Источник:** [BTS — Reporting Carrier On-Time Performance](https://www.transtats.bts.gov/PREZIP/)
- **Данные:** внутренние рейсы США за январь 2025 года
- **Первый разбор:** вылеты 15 января с учётом предыдущих рейсов тех же бортов

## 1. Загрузка данных

Загружаем из CSV 12 столбцов: дату рейса, номер борта, аэропорты,
плановое и фактическое время, задержки и признаки отмены и перенаправления.

In [81]:
import pandas as pd

from google.colab import drive

drive.mount("/content/drive")

csv_path = (
    "/content/drive/MyDrive/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_1.csv"
)

flights_raw = pd.read_csv(
    csv_path,
    usecols=[
        "FlightDate", "Tail_Number", "Origin", "Dest","CRSElapsedTime",
        "ActualElapsedTime","CRSDepTime", "DepTime", "CRSArrTime", "ArrTime",
        "DepDelay", "ArrDelay", "Cancelled", "Diverted",
    ],
)

assert len(flights_raw) == 539_747, f"Прочитано строк: {len(flights_raw):,}"

print(flights_raw.shape)
flights_raw.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(539747, 14)


,FlightDate,Tail_Number,Origin,Dest,CRSDepTime,DepTime,DepDelay,CRSArrTime,ArrTime,ArrDelay,Cancelled,Diverted,CRSElapsedTime,ActualElapsedTime
0,2025-01-01,N104NN,JFK,LAX,659,656.0,-3.0,1020,1013.0,-7.0,0.0,0.0,381.0,377.0
1,2025-01-02,N110AN,JFK,LAX,659,652.0,-7.0,1020,1022.0,2.0,0.0,0.0,381.0,390.0
2,2025-01-03,N106NN,JFK,LAX,659,652.0,-7.0,1020,1003.0,-17.0,0.0,0.0,381.0,371.0
3,2025-01-04,N117AN,JFK,LAX,700,653.0,-7.0,1026,1016.0,-10.0,0.0,0.0,386.0,383.0
4,2025-01-05,N104NN,JFK,LAX,659,655.0,-4.0,1020,1013.0,-7.0,0.0,0.0,381.0,378.0


## 2. Рейсы одного борта

Выберем самолёт, который есть в данных за 14 и 15 января.
Посмотрим, откуда и куда он летал, и сравним плановое и фактическое время.

In [82]:
sample_flights = flights_raw.loc[flights_raw["Tail_Number"] == "196NV"]
sample_flights = sample_flights.loc[
    sample_flights["FlightDate"].isin(["2025-01-14", "2025-01-15"])
].sort_values(["FlightDate", "CRSDepTime"])

sample_flights

,FlightDate,Tail_Number,Origin,Dest,CRSDepTime,DepTime,DepDelay,CRSArrTime,ArrTime,ArrDelay,Cancelled,Diverted,CRSElapsedTime,ActualElapsedTime
491249,2025-01-14,196NV,FNT,SRQ,1000,959.0,-1.0,1319,1343.0,24.0,0.0,0.0,199.0,224.0
487675,2025-01-14,196NV,SRQ,FNT,1409,1431.0,22.0,1712,1713.0,1.0,0.0,0.0,183.0,162.0
487894,2025-01-15,196NV,FNT,PGD,900,855.0,-5.0,1218,1210.0,-8.0,0.0,0.0,198.0,195.0
487583,2025-01-15,196NV,PGD,FNT,1308,1314.0,6.0,1610,1608.0,-2.0,0.0,0.0,182.0,174.0


## 3. Проверка полей времени

Проверим плановое и фактическое время вылета и прилёта.
Посчитаем пропуски, записи со временем 24:00 и значения вне формата HHMM.

In [83]:
flight_times = flights_raw[["CRSDepTime", "DepTime", "CRSArrTime", "ArrTime"]]

invalid_times = (
    (flight_times < 0)
    | (flight_times > 2400)
    | (flight_times % 100 >= 60)
    | (flight_times % 1 > 0)
)

time_quality = pd.DataFrame({
    "missing_count": flight_times.isna().sum(),
    "midnight_2400_count": (flight_times == 2400).sum(),
    "invalid_count": invalid_times.sum(),
})

time_quality

,missing_count,midnight_2400_count,invalid_count
CRSDepTime,0,0,0
DepTime,15886,33,0
CRSArrTime,0,0,0
ArrTime,16580,245,0


## 4. Перевод времени в минуты

Добавим четыре столбца с временем в минутах от начала суток.
Исходные значения оставим рядом, чтобы можно было сверить результат.

In [84]:
flight_minutes = (flight_times // 100) * 60 + flight_times % 100

flight_minutes = flight_minutes.astype("Int64").rename(columns={
    "CRSDepTime": "scheduled_departure_minutes",
    "DepTime": "actual_departure_minutes",
    "CRSArrTime": "scheduled_arrival_minutes",
    "ArrTime": "actual_arrival_minutes",
})

flights_clean = flights_raw.join(flight_minutes)

In [85]:
flights_clean.loc[flights_clean["DepTime"] == 2400].head()

,FlightDate,Tail_Number,Origin,Dest,CRSDepTime,DepTime,DepDelay,CRSArrTime,ArrTime,ArrDelay,Cancelled,Diverted,CRSElapsedTime,ActualElapsedTime,scheduled_departure_minutes,actual_departure_minutes,scheduled_arrival_minutes,actual_arrival_minutes
1879,2025-01-10,N776XF,MIA,JAX,2255,2400.0,65.0,23,116.0,53.0,0.0,0.0,88.0,76.0,1375,1440,23,76
10910,2025-01-18,N471AN,PHX,CLT,2358,2400.0,2.0,550,526.0,-24.0,0.0,0.0,232.0,206.0,1438,1440,350,326
15946,2025-01-16,N9010R,MIA,BNA,2246,2400.0,74.0,15,113.0,58.0,0.0,0.0,149.0,133.0,1366,1440,15,73
33845,2025-01-24,N424AN,LAX,DFW,2359,2400.0,1.0,505,447.0,-18.0,0.0,0.0,186.0,167.0,1439,1440,305,287
33846,2025-01-25,N435AN,LAX,DFW,2359,2400.0,1.0,505,429.0,-36.0,0.0,0.0,186.0,149.0,1439,1440,305,269


## 5. Дата и время вылета

Соберём плановый вылет из даты рейса и времени по расписанию.
Прибавим задержку и сверим полученное время с DepTime.

In [86]:
flight_dates = pd.to_datetime(flights_clean["FlightDate"], format="%Y-%m-%d")
scheduled_minutes = flights_clean["scheduled_departure_minutes"]

flights_clean["scheduled_departure_local"] = (
    flight_dates + pd.to_timedelta(scheduled_minutes, unit="min")
)

flights_clean["actual_departure_local"] = (
    flights_clean["scheduled_departure_local"]
    + pd.to_timedelta(flights_clean["DepDelay"], unit="min")
)

flights_clean.loc[1879, "actual_departure_local"]

Timestamp('2025-01-11 00:00:00')

In [87]:
actual_departure = flights_clean["actual_departure_local"]
calculated_minutes = actual_departure.dt.hour * 60 + actual_departure.dt.minute
reported_minutes = flights_clean["actual_departure_minutes"] % 1440

difference = (calculated_minutes - reported_minutes).dropna()

print("Сверили рейсов:", len(difference))
print("Расхождений:", (difference != 0).sum())

Сверили рейсов: 523824
Расхождений: 0


In [88]:
print("Строк в flights_raw:", len(flights_raw))
print("Строк в flights_clean:", len(flights_clean))

flights_clean[[
    "FlightDate",
    "DepTime",
    "DepDelay",
    "scheduled_departure_local",
    "actual_departure_local",
]].isna().sum()

Строк в flights_raw: 539747
Строк в flights_clean: 539747


,0
FlightDate,0
DepTime,15886
DepDelay,15923
scheduled_departure_local,0
actual_departure_local,15923


In [89]:
print(len(pd.read_csv(csv_path, usecols=["FlightDate"])))

539747


## 6. Часовые пояса аэропортов

Берём часовые пояса из OpenFlights. Добавляем вручную BIH, EAR и XWA, затем проверяем, что у всех аэропортов вылета и прилёта есть пояс.

In [90]:
airports_url = (
    "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat"
)

airport_timezones = pd.read_csv(
    airports_url,
    header=None,
    usecols=[4, 11],
    na_values=r"\N",
)
airport_timezones.columns = ["airport", "timezone"]
airport_timezones = airport_timezones.dropna().drop_duplicates()

flight_airports = pd.concat([flights_raw["Origin"], flights_raw["Dest"]]).unique()
airport_timezones = airport_timezones[
    airport_timezones["airport"].isin(flight_airports)
]

missing_airports = sorted(set(flight_airports) - set(airport_timezones["airport"]))

print("Аэропорты без часового пояса:", missing_airports)
print("Повторные коды:", airport_timezones["airport"].duplicated().sum())

Аэропорты без часового пояса: ['BIH', 'EAR', 'XWA']
Повторные коды: 0


In [91]:
timezone_by_airport = airport_timezones.set_index("airport")["timezone"].to_dict()
timezone_by_airport.update({
    "BIH": "America/Los_Angeles",
    "EAR": "America/Chicago",
    "XWA": "America/Chicago",
})

flights_clean["origin_timezone"] = flights_clean["Origin"].map(timezone_by_airport)
flights_clean["dest_timezone"] = flights_clean["Dest"].map(timezone_by_airport)

flights_clean[["origin_timezone", "dest_timezone"]].isna().sum()

,0
origin_timezone,0
dest_timezone,0


## 7. Время вылета в UTC

Переводим плановый вылет в UTC по часовому поясу аэропорта и прибавляем задержку. На примере рейса из Майами проверяем вылет после полуночи.

In [92]:
flights_clean["scheduled_departure_utc"] = pd.Series(
    pd.NaT, index=flights_clean.index, dtype="datetime64[ns, UTC]"
)

for timezone, flights in flights_clean.groupby("origin_timezone"):
    flights_clean.loc[flights.index, "scheduled_departure_utc"] = (
        flights["scheduled_departure_local"]
        .dt.tz_localize(timezone)
        .dt.tz_convert("UTC")
    )

flights_clean["actual_departure_utc"] = (
    flights_clean["scheduled_departure_utc"]
    + pd.to_timedelta(flights_clean["DepDelay"], unit="min")
)

## 8. Время прилёта в UTC

Прибавляем к вылету длительность рейса и сверяем задержку прилёта с `ArrDelay`. Из 522 269 строк в четырёх значения разошлись.

In [93]:
flights_clean["scheduled_arrival_utc"] = (
    flights_clean["scheduled_departure_utc"]
    + pd.to_timedelta(flights_clean["CRSElapsedTime"], unit="min")
)

flights_clean["actual_arrival_utc"] = (
    flights_clean["actual_departure_utc"]
    + pd.to_timedelta(flights_clean["ActualElapsedTime"], unit="min")
)

arrival_delay = (
    flights_clean["actual_arrival_utc"] - flights_clean["scheduled_arrival_utc"]
).dt.total_seconds() / 60

reported_delay = flights_clean["ArrDelay"]
can_compare = arrival_delay.notna() & reported_delay.notna()

print("Прилётов сверено:", can_compare.sum())
print("Расхождений с ArrDelay:", (arrival_delay[can_compare] != reported_delay[can_compare]).sum())

Прилётов сверено: 522269
Расхождений с ArrDelay: 4


In [94]:
flights_clean.loc[1879, [
    "Origin",
    "origin_timezone",
    "actual_departure_local",
    "scheduled_departure_utc",
    "actual_departure_utc",
]]

,1879
Origin,MIA
origin_timezone,America/New_York
actual_departure_local,2025-01-11 00:00:00
scheduled_departure_utc,2025-01-11 03:55:00+00:00
actual_departure_utc,2025-01-11 05:00:00+00:00


In [95]:
arrival_clock_minutes = pd.Series(pd.NA, index=flights_clean.index, dtype="Int64")

for timezone, flights in flights_clean.groupby("dest_timezone"):
    arrival_local = flights["actual_arrival_utc"].dt.tz_convert(timezone)
    arrival_clock_minutes.loc[flights.index] = (
        arrival_local.dt.hour * 60 + arrival_local.dt.minute
    ).astype("Int64")

reported_arrival_minutes = flights_clean["actual_arrival_minutes"] % 1440
clock_difference = (arrival_clock_minutes - reported_arrival_minutes).dropna()
clock_mismatches = clock_difference[clock_difference != 0]

print("Прилётов сверено:", len(clock_difference))
print("Расхождений с ArrTime:", len(clock_mismatches))

Прилётов сверено: 522269
Расхождений с ArrTime: 8


In [96]:
delay_mismatches = arrival_delay[can_compare & (arrival_delay != reported_delay)]

both_mismatches = delay_mismatches.index.intersection(clock_mismatches.index)
arrival_mismatches = delay_mismatches.index.union(clock_mismatches.index)

flights_clean["arrival_mismatch"] = flights_clean.index.isin(arrival_mismatches)

print("Разошлись обе проверки:", len(both_mismatches))
print("Всего спорных прилётов:", len(arrival_mismatches))

Разошлись обе проверки: 2
Всего спорных прилётов: 10


## 9. Соседние рейсы одного борта

На примере борта `196NV` проверим, как связать соседние рейсы и посчитать время на земле между прилётом и следующим вылетом.

In [97]:
tail_flights = flights_clean.loc[
    (flights_clean["Tail_Number"] == "196NV")
    & flights_clean["FlightDate"].isin(["2025-01-14", "2025-01-15"])
].sort_values("actual_departure_utc").copy()

tail_flights["previous_dest"] = tail_flights["Dest"].shift()
same_airport = tail_flights["previous_dest"] == tail_flights["Origin"]

ground_time = (
    tail_flights["actual_departure_utc"] - tail_flights["actual_arrival_utc"].shift()
)
tail_flights["ground_minutes"] = (ground_time.dt.total_seconds() / 60).where(same_airport)

tail_flights[["FlightDate", "Origin", "Dest", "previous_dest", "ground_minutes"]]

,FlightDate,Origin,Dest,previous_dest,ground_minutes
491249,2025-01-14,FNT,SRQ,None,NaN
487675,2025-01-14,SRQ,FNT,SRQ,48.0
487894,2025-01-15,FNT,PGD,FNT,942.0
487583,2025-01-15,PGD,FNT,PGD,64.0


In [98]:
flight_pairs = flights_clean.dropna(
    subset=["Tail_Number", "actual_departure_utc"]
).sort_values(["Tail_Number", "actual_departure_utc"]).copy()

valid_arrival = (
    (flight_pairs["Cancelled"] == 0)
    & (flight_pairs["Diverted"] == 0)
    & (flight_pairs["arrival_mismatch"] == False)
)
flight_pairs["arrival_for_pairing"] = flight_pairs["actual_arrival_utc"].where(valid_arrival)

by_tail = flight_pairs.groupby("Tail_Number", sort=False)
flight_pairs["previous_dest"] = by_tail["Dest"].shift()
flight_pairs["previous_arrival_utc"] = by_tail["arrival_for_pairing"].shift()

ground_minutes = (
    flight_pairs["actual_departure_utc"] - flight_pairs["previous_arrival_utc"]
).dt.total_seconds() / 60

same_airport = flight_pairs["previous_dest"] == flight_pairs["Origin"]
flight_pairs["ground_minutes"] = ground_minutes.where(
    same_airport & (ground_minutes >= 0) & (flight_pairs["Cancelled"] == 0)
)

jan15_pairs = flight_pairs.loc[flight_pairs["FlightDate"] == "2025-01-15"]

print("Стоянок рассчитано за 15 января:", jan15_pairs["ground_minutes"].count())
jan15_pairs.loc[
    jan15_pairs["Tail_Number"] == "196NV",
    ["Origin", "Dest", "previous_dest", "ground_minutes"],
]

Стоянок рассчитано за 15 января: 15841


,Origin,Dest,previous_dest,ground_minutes
487894,FNT,PGD,FNT,942.0
487583,PGD,FNT,PGD,64.0


## 10. Плановая и фактическая стоянка

Для каждой пары рейсов сравним время на земле по расписанию и по факту. Добавим задержку предыдущего прилёта, чтобы дальше сопоставить её с задержкой следующего вылета.

In [99]:
previous_scheduled_arrival = by_tail["scheduled_arrival_utc"].shift()
flight_pairs["previous_arrival_delay"] = by_tail["ArrDelay"].shift()

scheduled_ground = (
    flight_pairs["scheduled_departure_utc"] - previous_scheduled_arrival
).dt.total_seconds() / 60

flight_pairs["scheduled_ground_minutes"] = scheduled_ground.where(
    flight_pairs["ground_minutes"].notna() & (scheduled_ground >= 0)
)

jan15_pairs = flight_pairs.loc[flight_pairs["FlightDate"] == "2025-01-15"]

jan15_pairs.loc[
    jan15_pairs["Tail_Number"] == "196NV",
    [
        "Origin", "Dest", "scheduled_ground_minutes", "ground_minutes",
        "previous_arrival_delay", "DepDelay",
    ],
]

,Origin,Dest,scheduled_ground_minutes,ground_minutes,previous_arrival_delay,DepDelay
487894,FNT,PGD,948.0,942.0,1.0,-5.0
487583,PGD,FNT,50.0,64.0,-8.0,6.0


## 11. Задержка следующего вылета

Для рейсов с плановой датой вылета 15 января сравниваем долю вылетов с задержкой от 15 минут. Смотрим отдельно на пары со стоянкой до часа и от одного до трёх часов по расписанию. В каждой группе сравниваем рейсы после прилётов с задержкой от 15 минут и после остальных прилётов.

In [100]:
turnarounds = jan15_pairs.loc[
    (jan15_pairs["scheduled_ground_minutes"] > 0)
    & (jan15_pairs["scheduled_ground_minutes"] <= 180)
    & jan15_pairs["previous_arrival_delay"].notna()
].copy()

turnarounds["scheduled_ground_group"] = pd.cut(
    turnarounds["scheduled_ground_minutes"],
    bins=[0, 60, 180],
    labels=["До 60 минут", "Свыше 60 и до 180 минут"],
)
turnarounds["previous_arrival_delayed"] = turnarounds["previous_arrival_delay"] >= 15
turnarounds["departure_delayed"] = turnarounds["DepDelay"] >= 15

delay_summary = (
    turnarounds
    .groupby(["scheduled_ground_group", "previous_arrival_delayed"], observed=True)
    .agg(
        pairs=("departure_delayed", "size"),
        delayed_departures=("departure_delayed", "sum"),
    )
)

delay_summary["delayed_departure_pct"] = (
    100 * delay_summary["delayed_departures"] / delay_summary["pairs"]
).round(1)

delay_summary

pairs  delayed_departures  \
scheduled_ground_group  previous_arrival_delayed                              
До 60 минут             False                      5892                 229   
                        True                        333                 241   
Свыше 60 и до 180 минут False                      4334                 140   
                        True                        347                 111   

                                                  delayed_departure_pct  
scheduled_ground_group  previous_arrival_delayed                         
До 60 минут             False                                       3.9  
                        True                                       72.4  
Свыше 60 и до 180 минут False                                       3.2  
                        True                                       32.0

## 12. Выгрузка данных для PostgreSQL

Выгружаем январские рейсы с временем в UTC и флагом спорного прилёта. В PostgreSQL заново находим соседние рейсы каждого борта и сверяем результат за 15 января с расчётом в pandas.

In [101]:
sql_columns = [
    "FlightDate", "Tail_Number", "Origin", "Dest",
    "scheduled_departure_utc", "actual_departure_utc",
    "scheduled_arrival_utc", "actual_arrival_utc",
    "DepDelay", "ArrDelay", "Cancelled", "Diverted", "arrival_mismatch",
]

sql_export = flights_clean[sql_columns].copy()
sql_export.columns = sql_export.columns.str.lower()
sql_export.insert(0, "flight_row_id", flights_clean.index)

sql_csv_path = "/content/drive/MyDrive/flights_2025_01_utc.csv"
sql_export.to_csv(sql_csv_path, index=False)

print("Строк в выгрузке:", len(sql_export))
print(sql_csv_path)

Строк в выгрузке: 539747
/content/drive/MyDrive/flights_2025_01_utc.csv


## 13. Сверка с PostgreSQL

В PostgreSQL заново собрал пары рейсов одного борта через `LAG`. Для вылетов с плановой датой 15 января 2025 года результаты совпали с pandas во всех четырёх группах.

При плановой стоянке до 60 минут следующий вылет задерживался на 15 минут и больше в 72,4% случаев после позднего прилёта и в 3,9% после остальных прилётов. При стоянке свыше 60 и до 180 минут доли составили 32,0% и 3,2%. В этом срезе связь между задержками была сильнее при короткой стоянке.